In [0]:
from datetime import date, timedelta
import os
import json
import urllib.request
import urllib.parse
from pyspark.sql import functions as F

catalogo = "cambio"

# Janela móvel curta para carga incremental
data_fim = date.today()
data_inicio = data_fim - timedelta(days=5)

data_inicio_api = data_inicio.strftime("%m-%d-%Y")
data_fim_api = data_fim.strftime("%m-%d-%Y")

data_inicio_ref = data_inicio.strftime("%Y-%m-%d")
data_fim_ref = data_fim.strftime("%Y-%m-%d")

batch_id = f"{data_inicio_ref}_{data_fim_ref}".replace("-", "")

moedas = ["USD", "EUR", "GBP", "JPY", "CAD", "AUD"]

landing_dir = f"/Volumes/{catalogo}/bronze/landing/ptax/batch_{batch_id}"
os.makedirs(landing_dir, exist_ok=True)

print(f"Período: {data_inicio_ref} até {data_fim_ref}")

In [0]:
def buscar_odata_paginado(url_base: str):
    registros = []
    url = url_base
    pagina = 1

    while url:
        print(f"Buscando página {pagina}: {url}")

        with urllib.request.urlopen(url, timeout=120) as response:
            payload = json.loads(response.read().decode("utf-8"))

        valores = payload.get("value", [])
        registros.extend(valores)

        url = payload.get("@odata.nextLink")
        pagina += 1

    return registros

In [0]:
for moeda in moedas:
    endpoint = (
        "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
        f"CotacaoMoedaPeriodo(moeda=@moeda,dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    )

    parametros = {
        "@moeda": f"'{moeda}'",
        "@dataInicial": f"'{data_inicio_api}'",
        "@dataFinalCotacao": f"'{data_fim_api}'",
        "$format": "json",
        "$top": "100"
    }

    url = endpoint + "?" + urllib.parse.urlencode(parametros, safe="'@$")

    registros = buscar_odata_paginado(url)

    payload_saida = {
        "moeda": moeda,
        "data_inicio": data_inicio_ref,
        "data_fim": data_fim_ref,
        "batch_id": batch_id,
        "qtd_registros": len(registros),
        "registros": registros
    }

    arquivo_saida = f"{landing_dir}/ptax_{moeda}_{data_inicio_ref}_{data_fim_ref}.json"

    with open(arquivo_saida, "w", encoding="utf-8") as f:
        json.dump(payload_saida, f, ensure_ascii=False)

    print(f"Moeda {moeda}: {len(registros)} registros salvos em {arquivo_saida}")

In [0]:
df_bronze = (
    spark.read
    .option("multiLine", "true")
    .json(f"{landing_dir}/*.json")
    .select(
        "*",
        F.col("_metadata.file_path").alias("_arquivo_lido")
    )
    .withColumn("_data_ingestao", F.current_timestamp())
)

df_bronze.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalogo}.bronze.ptax_raw")

In [0]:
df_bronze = spark.table("cambio.bronze.ptax_raw")

display(
    df_bronze.select(
        "moeda",
        "data_inicio",
        "data_fim",
        "batch_id",
        "qtd_registros",
        "_arquivo_lido",
        "_data_ingestao"
    )
)

display(
    df_bronze
    .groupBy("batch_id", "moeda")
    .agg(F.max("qtd_registros").alias("qtd_registros"))
    .orderBy("moeda")
)